In [46]:
import plotly.graph_objects as go
import numpy as np
import json
import pandas as pd

In [ ]:
base_path = "results"

model = "google__embeddinggemma-300m"
#model = "minishlab__potion-base-8M"
#model = "intfloat__multilingual-e5-large-instruct"
#model= "intfloat__multilingual-e5-small"
#model = "BAAI__bge-m3"
#model = "Qwen__Qwen3-Embedding-0.6B"
dataset = "tatoeba:fin-eng"
path =f"{base_path}/{model}/{dataset}/"

In [48]:
scores_path= path+"mrr@20__Instruct-Query-template.json"
with open(scores_path, "r") as f:
    scores = json.load(f)

In [49]:
with open(path+"prompt_data_cosine_query_to_prompt.json") as f:
    data1 = json.load(f) 

with open(path+"prompt_data_cosine_query_to_answer.json") as f:
    data2 = json.load(f) 


In [50]:
df_scores = pd.DataFrame.from_dict(scores, orient="index", columns=["score"]) #columns= [f"prompt{i}" for i in range(len(scores.values()))])
df_scores = df_scores.reset_index().rename(columns={"index": "prompt_text"})


In [51]:
df_prompts = []

df_prompt = pd.DataFrame.from_dict(data1).T
df_answer = pd.DataFrame.from_dict(data2).T



In [52]:
import plotly.graph_objects as go
import numpy as np

# Get alpha columns (all columns except 'prompt_text')
alpha_values = [col for col in df_prompt.columns if col != 'prompt_text']

# Merge all dataframes on prompt_text
import pandas as pd
df = df_scores.merge(df_prompt, on='prompt_text').merge(
    df_answer, on='prompt_text', suffixes=('_prompt', '_answer')
)

# Normalize scores for marker size (plotly needs positive sizes)
min_size, max_size = 5, 30
scores = df['score']
#marker_sizes = (
#    (scores - scores.min()) / (scores.max() - scores.min()) * (max_size - min_size) + min_size
#)
ranks = scores.rank(method='average')
marker_sizes = (
    (ranks - ranks.min()) / (ranks.max() - ranks.min()) * (max_size - min_size) + min_size
)

# Create figure
fig = go.Figure()

# Add one trace per alpha value
for alpha in alpha_values:
    x_vals = df[f'{alpha}_prompt'].apply(lambda d: float(d['mean']))
    y_vals = df[f'{alpha}_answer'].apply(lambda d: float(d['mean']))
    x_err  = df[f'{alpha}_prompt'].apply(lambda d: float(d['std']))
    y_err  = df[f'{alpha}_answer'].apply(lambda d: float(d['std']))

    fig.add_trace(
        go.Scatter(
            visible=False,
            mode='markers',
            x=x_vals,
            y=y_vals,
            error_x=dict(type='data', array=x_err, visible=True, color='lightgray'),  # optional std bars
            error_y=dict(type='data', array=y_err, visible=True, color='lightgray'),
            marker=dict(
                size=marker_sizes,
                colorscale='Cividis',
                color=scores,
                colorbar=dict(title='Score'),
                showscale=True,
            ),
            text=df['prompt_text'],
            hovertemplate=(
                '<b>Prompt:</b> %{text}<br>'
                '<b>X (prompt mean):</b> %{x:.4f}<br>'
                '<b>Y (answer mean):</b> %{y:.4f}<br>'
                '<b>Score:</b> %{marker.color:.4f}<extra></extra>'
            ),
            name=f'α = {alpha}',
        )
    )


# Make first trace visible
fig.data[0].visible = True

# Build slider steps
steps = []
for i, alpha in enumerate(alpha_values):
    step = dict(
        method='update',
        label=str(alpha),
        args=[
            {'visible': [False] * len(fig.data)},
            {'title': f'Alpha: {alpha}'},
        ],
    )
    step['args'][0]['visible'][i] = True
    steps.append(step)

sliders = [dict(
    active=0,
    currentvalue={'prefix': 'Alpha: ', 'visible': True, 'xanchor': 'center'},
    pad={'t': 50},
    steps=steps,
)]

fig.update_layout(
    sliders=sliders,
    title=f'Alpha: {alpha_values[0]}',
    xaxis_title='Cos-distance to Query-Prompt line (α = 0 -> Query)',
    yaxis_title='Cos-distance to Query-Answer line (α = 0 -> Query)',
    height=600,
)

all_x, all_y = [], []
for alpha in alpha_values:
    all_x.append(df[f'{alpha}_prompt'].apply(lambda d: float(d['mean'])))
    all_y.append(df[f'{alpha}_answer'].apply(lambda d: float(d['mean'])))

xmin = min(s.min() for s in all_x)-0.1
xmax = max(s.max() for s in all_x)+0.1
ymin = min(s.min() for s in all_y)-0.1
ymax = max(s.max() for s in all_y)+0.1

fig.update_layout(
    xaxis=dict(range=[xmin, xmax], autorange=False),
    yaxis=dict(range=[ymin, ymax], autorange=False),
)


fig.add_annotation(x=xmax-0.05, y=ymax-0.05,
            text="Close to both",
            showarrow=False,
            )
fig.add_annotation(x=xmax-0.05, y=ymin+0.05,
            text="Close to prompt",
            showarrow=False,
            )
fig.add_annotation(x=xmin+0.05, y=ymax-0.05,
            text="Close to answer",
            showarrow=False,
            )
fig.show()